# Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

# Read from Silver

fact_sales spaja:
- `crm_sales_details` — transakcije: order number, dates, sales, quantity, price
- `gold.dim_customers`  — customer_key (surrogate key)
- `gold.dim_products`   — product_key_sk (surrogate key)

Fact tabela cuva **surrogate key-eve** iz dim tabela, ne prirodne kljuceve.
Ovo je osnova star sheme.

In [0]:
sales      = spark.table("workspace.silver.crm_sales_details")
dim_cust   = spark.table("workspace.gold.dim_customers")
dim_prod   = spark.table("workspace.gold.dim_products")

print("=== crm_sales_details ===")
sales.printSchema()
sales.show(3, truncate=False)

# Data Quality Checks

Pre bilo kakve transformacije — proveravamo kvalitet podataka.  
Ovo je **tema 39 — Data Quality**.

Tipicni problemi u sales podacima:
- `sales != quantity * price` (nekonzistentnost)
- Negativne cene ili kolicine
- `order_date > ship_date` (logicka greska)
- NULL vrednosti u obaveznim kolonama

Biljezimo probleme, ne bacamo podatke — to odlucuje biznis, ne inzenjer.

In [0]:
# DQ Check 1: Konzistentnost sales = quantity * price
# Ako sales = 0 ali quantity i price nisu 0 -> koristimo izracunatu vrednost

print("--- DQ Check: sales vs quantity*price ---")
(
    sales
    .withColumn("expected_sales", F.col("quantity") * F.col("price"))
    .filter(
        (F.col("sales") != F.col("expected_sales")) |
        (F.col("sales").isNull()) |
        (F.col("sales") <= 0)
    )
    .select("sales_order_number", "sales", "quantity", "price", "expected_sales")
    .show(10)
)

# DQ Check 2: Negativne vrednosti
print("--- DQ Check: negativne cene/kolicine ---")
sales.filter(
    (F.col("price") < 0) | (F.col("quantity") <= 0)
).select("sales_order_number", "quantity", "price").show(5)

# DQ Check 3: NULL-ovi u kljucnim kolonama
print("--- DQ Check: NULL u kljucnim kolonama ---")
for col_name in ["sales_order_number", "sales_product_key", "sales_customer_id"]:
    null_count = sales.filter(F.col(col_name).isNull()).count()
    print(f"  {col_name}: {null_count} NULL-ova")

# Data Transformations

## Date transformacija

Datumi u source sistemu cesto dolaze kao INT (YYYYMMDD) ili STRING.
Pretvaramo ih u proper DateType.

## Sales fix

Ako `sales` je 0 ili NULL ali `quantity` i `price` postoje —  
racunamo `sales = quantity * price`.

In [0]:
# Korak 1: Date transformacija
# Source moze imati INT format YYYYMMDD ili STRING

def fix_date_col(df, col_name):
    """Konvertuje INT YYYYMMDD ili STRING u DateType.
    Ako je van validnog opsega (npr. 0, negativan) -> NULL.
    """
    return df.withColumn(
        col_name,
        F.when(
            # Eksplicitno odbaci nevažeće vrednosti: NULL, 0, prazan string
            (F.col(col_name).isNull()) | 
            (F.col(col_name) == 0) |
            (F.col(col_name).cast("string") == "0") |
            (F.col(col_name).cast("string") == ""),
            F.lit(None).cast("date")
        ).when(
            # YYYYMMDD format (8 cifara)
            F.col(col_name).cast("string").rlike(r'^\d{8}$'),
            F.to_date(F.col(col_name).cast("string"), "yyyyMMdd")
        ).otherwise(
            # Pokušaj default konverziju, ali ignoriši greške
            F.to_date(F.col(col_name).cast("string"))
        )
    )

sales_clean = sales
for date_col in ["sales_order_date", "sales_ship_date", "sales_due_date"]:
    if date_col in sales.columns:
        sales_clean = fix_date_col(sales_clean, date_col)

In [0]:
# Korak 2: Fix sales vrednosti
# Ako sales je 0 ili NULL -> quantity * price
# Ako price je 0 ili NULL -> sales / quantity

sales_clean = (
    sales_clean
    .withColumn(
        "price",
        F.when(
            (F.col("price").isNull()) | (F.col("price") <= 0),
            F.col("sales") / F.col("quantity")
        ).otherwise(F.col("price"))
    )
    .withColumn(
        "sales",
        F.when(
            (F.col("sales").isNull()) | (F.col("sales") <= 0),
            F.col("quantity") * F.col("price")
        ).otherwise(F.col("sales"))
    )
)

In [0]:
# Korak 3: JOIN sa dim tabela da dobijemo surrogate key-eve

# Normalizacija sales_customer_id za join sa dim_customers
sales_clean = sales_clean.withColumn(
    "cust_id_normalized",
    F.regexp_replace(F.col("sales_customer_id"), r'NAS-', '')
)

# Dim customers: uzimamo samo customer_key + customer_id za join
cust_keys = dim_cust.select(
    F.col("customer_key"),
    F.regexp_replace(F.col("customer_id"), r'NAS-', '').alias("cust_join_key")
)

# Dim products: samo aktuelni proizvodi (is_current = true) za join
prod_keys = dim_prod.filter(F.col("is_current") == True).select(
    F.col("product_key_sk"),
    F.col("product_key").alias("prod_join_key")
)

fact_sales = (
    sales_clean
    .join(
        cust_keys,
        sales_clean["cust_id_normalized"] == cust_keys["cust_join_key"],
        how="left"
    )
    .join(
        prod_keys,
        sales_clean["sales_product_key"] == prod_keys["prod_join_key"],
        how="left"
    )
    .select(
        F.col("sales_order_number"),
        F.col("customer_key"),
        F.col("product_key_sk"),
        F.col("sales_order_date").alias("order_date"),
        F.col("sales_ship_date").alias("ship_date"),
        F.col("sales_due_date").alias("due_date"),
        F.col("sales"),
        F.col("quantity"),
        F.col("price")
    )
)

fact_sales.display()

# Write to Gold

In [0]:
(
    fact_sales.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("workspace.gold.fact_sales")
)

# OPTIMIZE + ZORDER

## Small Files Problem

Svaki write (posebno streaming ili cest batch) kreira nove Parquet fajlove.
Nakon vremena imas hiljade malih fajlova:
- Svaki file open = overhead (S3/ADLS API pozivi)
- Vise fajlova = sporije skeniranje
- Primer: 10.000 fajlova po 1MB = sporije od 10 fajlova po 1GB

## OPTIMIZE (bin packing)

Kombinuje male fajlove u optimalan broj vecih fajlova (~1GB target).
Ne menja logicke podatke, samo fizicku organizaciju.

## ZORDER

Co-locates slicne vrednosti u istim fajlovima.  
Kombinijes sa OPTIMIZE:
```sql
OPTIMIZE tabela ZORDER BY (kolona)
```
Ako uvek filtriras po `order_date` i `customer_key` —  
ZORDER po tim kolonama znaci da Spark preskace vise fajlova (data skipping).

**Pravilo:** ZORDER je efikasan na kolonama sa **visokom kardinalnoscu**  
po kojima cesto filtriras. Ne stavljaj ZORDER na boolean ili gender kolonu.

In [0]:
%sql
-- Koliko fajlova ima tabela pre OPTIMIZE?
DESCRIBE DETAIL workspace.gold.fact_sales

In [0]:
%sql
-- OPTIMIZE + ZORDER
-- Fact tabele se najcesce filtriraju po datumu i customer/product
-- Za fact_sales: order_date i customer_key su prirodni kandidati
OPTIMIZE workspace.gold.fact_sales
ZORDER BY (order_date, customer_key)

In [0]:
%sql
-- Posle OPTIMIZE: manji broj fajlova, bolje organizovani
DESCRIBE DETAIL workspace.gold.fact_sales

# Auto Optimize — autoCompact i optimizeWrite

Umesto da rucno pokreces OPTIMIZE, mozes da ukljucis automatski:

**optimizeWrite** — pri svakom write-u Spark automatski bira optimalan  
broj output fajlova (smanjuje shuffle particije ako su podaci mali).  
Dodaje malo latency na write, ali sprecava small files problem.

**autoCompact** — posle svakog write-a, Databricks u pozadini  
kompaktuje fajlove koji su manji od target velicine.  
Asinhrono, ne blokira write.

Kada koristiti:
- Streaming tabele → uvek ukljuci oba
- Batch sa cescim write-ovima → ukljuci oba
- Retki veliki batch load → nije potrebno

In [0]:
%sql
ALTER TABLE workspace.gold.fact_sales
SET TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact'   = 'true'
)

# VACUUM — ciscenje starih fajlova

Delta Lake **ne brise** stare Parquet fajlove odmah posle OPTIMIZE ili write-a.  
Oni ostaju na disku da bi **Time Travel** radio (mozete citati stare verzije).

VACUUM brise fajlove starije od `retentionHours` (default: 168h = 7 dana).

**Paznja:** Posle VACUUM ne mozes da radis Time Travel na obrisane verzije!

```sql
VACUUM tabela RETAIN 168 HOURS  -- brise fajlove starije od 7 dana
VACUUM tabela RETAIN 0 HOURS    -- brise SVE stare fajlove (opasno!)
```

Praksa: pokreni VACUUM jednom nedeljno u okviru maintenance job-a.

In [0]:
%sql
-- DRY RUN: pokaze koji fajlovi bi bili obrisani, ali NE brise ih
VACUUM workspace.gold.fact_sales DRY RUN

In [0]:
%sql
-- Time Travel pre VACUUM
SELECT COUNT(*) as rows_v0 FROM workspace.gold.fact_sales VERSION AS OF 0

# Time Travel vs RESTORE

**Time Travel** — citas staru verziju, tabela se ne menja:
```sql
SELECT * FROM tabela VERSION AS OF 3
SELECT * FROM tabela TIMESTAMP AS OF '2024-01-01'
```
Koristis za: audit, debugging, uporedivanje verzija

**RESTORE** — vraca tabelu na staru verziju, MENJA tabelu:
```sql
RESTORE TABLE tabela TO VERSION AS OF 3
RESTORE TABLE tabela TO TIMESTAMP AS OF '2024-01-01'
```
Koristis za: recovery posle lose migracije, rollback bad write-a

Kljucna razlika: Time Travel = read-only pogled u proslost.  
RESTORE = fizicka promena stanja tabele.

In [0]:
%sql
-- Simulacija: napravimo 'gresku' — writujemo praznu tabelu
-- pa restorujemo na prethodnu verziju

-- Korak 1: pamtimo trenutni count
SELECT COUNT(*) as current_count FROM workspace.gold.fact_sales

In [0]:
# Korak 2: Simuliramo 'los write' — brisemo sve
spark.sql("DELETE FROM workspace.gold.fact_sales WHERE 1=1")
print("Los write simuliran.")
spark.sql("SELECT COUNT(*) as after_bad_write FROM workspace.gold.fact_sales").show()

In [0]:
%sql
-- Korak 3: DESCRIBE HISTORY da vidimo verzije
DESCRIBE HISTORY workspace.gold.fact_sales

In [0]:
%sql
-- Korak 4: RESTORE na verziju pre loseg write-a (VERSION AS OF 1 = posle OPTIMIZE)
-- Verzija 0 = initial write, Verzija 1 = posle OPTIMIZE, Verzija 2 = los DELETE
RESTORE TABLE workspace.gold.fact_sales TO VERSION AS OF 1

In [0]:
%sql
-- Korak 5: Verifikacija — podaci su se vratili
SELECT COUNT(*) as restored_count FROM workspace.gold.fact_sales

# Lazy Evaluation

Spark ne izvrsava transformacije odmah — samo gradi **execution plan**.
Akcija (`.count()`, `.show()`, `.write`) pokrece izvrsavanje.

```python
df = spark.table("...")        # NE citamo nista
df = df.filter(...)            # NE filtriramo nista, dodajemo u plan
df = df.join(...)              # NE joinujemo nista, dodajemo u plan
df.count()                     # SADA se sve izvrsava odjednom
```

Prednost: Spark optimizer (Catalyst) moze da optimizuje ceo plan
pre izvrsavanja — pushdown predikata, eliminacija kolona, reorder join-ova.

Zato pisemo transformacije kao chain, ne pozivamo .show() posle svake.

In [0]:
# Primer lazy evaluation: explain() pokazuje plan BEZ izvrsavanja
fact_sales.explain(extended=True)

# Temporary Views vs Global Temporary Views

**Temporary View** — vidljiva samo u trenutnoj Spark sesiji (trenutni cluster):
```python
df.createOrReplaceTempView("my_view")
spark.sql("SELECT * FROM my_view")
```
Nestaje kada se sesija zavrsi (restart cluster = view nestaje).

**Global Temporary View** — vidljiva svim sesijama na istom clusteru,  
u specijalnom `global_temp` schema:
```python
df.createOrReplaceGlobalTempView("my_global_view")
spark.sql("SELECT * FROM global_temp.my_global_view")
```
Koristis kada vise notebooks treba da dele isti privremeni DataFrame  
bez da ga pises kao Delta tabelu.

**Ni jedan ni drugi** ne cuvaju podatke — to su samo named queriji.

In [0]:
# Primer: kreiranje temp view za ad-hoc SQL analizu
fact_sales.createOrReplaceTempView("vw_fact_sales")
dim_cust.createOrReplaceTempView("vw_dim_customers")
dim_prod.filter(F.col("is_current") == True).createOrReplaceTempView("vw_dim_products")

In [0]:
%sql
-- Primer star schema query kroz temp views
SELECT
    c.country,
    p.category,
    YEAR(f.order_date)   AS godina,
    MONTH(f.order_date)  AS mesec,
    SUM(f.sales)         AS ukupna_prodaja,
    COUNT(DISTINCT f.sales_order_number) AS broj_porudzbina,
    AVG(f.sales)         AS prosecna_vrednost
FROM vw_fact_sales f
LEFT JOIN vw_dim_customers c ON f.customer_key   = c.customer_key
LEFT JOIN vw_dim_products  p ON f.product_key_sk = p.product_key_sk
GROUP BY c.country, p.category, YEAR(f.order_date), MONTH(f.order_date)
ORDER BY ukupna_prodaja DESC
LIMIT 20